# Bootstrap CI on the compound-validation results — no retraining needed

Loads the raw per-seed predictions already saved in the checkpoint files from the
Stage-4-v2 run (`m4_hybrid_original_seed*.pkl`, `m4_hybrid_v2_compoundval_seed*.pkl`)
and computes bootstrap 95% CIs for F1 on both variants, all three test sets. This is
pure post-hoc analysis on data that already exists — no GPU, no re-extraction, no
retraining.

In [ ]:
import os, glob, pickle
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
ARTIFACTS_DIR = '/kaggle/working/artifacts'

def ckpt_load(name):
    path = os.path.join(CHECKPOINT_DIR, name + '.pkl')
    with open(path, 'rb') as f:
        return pickle.load(f)

SEEDS = [42, 43, 44, 45, 46]
TEST_NAMES = ['testA', 'testB', 'testC']
VARIANTS = {'Original (reddit-val)': 'm4_hybrid_original', 'Compound-val (reddit+dolly)': 'm4_hybrid_v2_compoundval'}

# Verify all expected checkpoints actually exist before proceeding -- fail loudly and clearly
# rather than silently computing a CI on a partial/wrong set of seeds.
missing = []
for prefix in VARIANTS.values():
    for seed in SEEDS:
        name = f'{prefix}_seed{seed}'
        if not os.path.exists(os.path.join(CHECKPOINT_DIR, name + '.pkl')):
            missing.append(name)
if missing:
    raise FileNotFoundError(f'Missing {len(missing)} expected checkpoint(s), cannot proceed: {missing}')
print(f'All {len(VARIANTS)*len(SEEDS)} expected checkpoints found. Loading raw predictions...')


In [ ]:
def bootstrap_ci_f1(labels, preds, n_boot=3000, seed=42, alpha=0.05):
    rng = np.random.default_rng(seed)
    labels, preds = np.asarray(labels), np.asarray(preds)
    n = len(labels)
    stats = [f1_score(labels[idx], preds[idx], zero_division=0) for idx in (rng.integers(0, n, n) for _ in range(n_boot))]
    lo, hi = np.percentile(stats, [100*alpha/2, 100*(1-alpha/2)])
    return float(np.mean(stats)), float(lo), float(hi)

ci_rows = []
for variant_name, prefix in VARIANTS.items():
    for seed in SEEDS:
        seed_result = ckpt_load(f'{prefix}_seed{seed}')
        for test_name in TEST_NAMES:
            labels, preds = seed_result['raw'][test_name]
            mean_f1, lo, hi = bootstrap_ci_f1(labels, preds, seed=seed)
            ci_rows.append({'variant': variant_name, 'test_set': test_name, 'seed': seed,
                             'f1_point_estimate': f1_score(labels, preds, zero_division=0),
                             'f1_bootstrap_mean': mean_f1, 'ci_lower': lo, 'ci_upper': hi})

ci_df = pd.DataFrame(ci_rows).round(4)
print(ci_df.to_string(index=False))
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
ci_df.to_csv(f'{ARTIFACTS_DIR}/M4_v2_bootstrap_ci_per_seed.csv', index=False)


## Aggregate view — does the CI story change the headline claim?

For each variant/test combination, checks whether the 5 per-seed CIs (each already
accounting for within-test-set sampling uncertainty) are consistent with each other,
and reports the across-seed range for a fuller picture than a single mean±std line.

In [ ]:
summary_rows = []
for variant_name in VARIANTS:
    for test_name in TEST_NAMES:
        sub = ci_df[(ci_df.variant == variant_name) & (ci_df.test_set == test_name)]
        summary_rows.append({
            'variant': variant_name, 'test_set': test_name,
            'f1_mean_across_seeds': sub.f1_point_estimate.mean(),
            'f1_std_across_seeds': sub.f1_point_estimate.std(),
            'widest_ci_lower': sub.ci_lower.min(),
            'widest_ci_upper': sub.ci_upper.max(),
            'narrowest_ci_width': (sub.ci_upper - sub.ci_lower).min(),
            'widest_ci_width': (sub.ci_upper - sub.ci_lower).max(),
        })
summary_df = pd.DataFrame(summary_rows).round(4)
print(summary_df.to_string(index=False))
summary_df.to_csv(f'{ARTIFACTS_DIR}/M4_v2_bootstrap_ci_summary.csv', index=False)

print()
print('Read this as: does the compound-val CI range overlap the original-val CI range?')
print('If the ranges are cleanly separated (no overlap) for a test set, the improvement is')
print('robust to per-sample resampling noise on top of already being robust to seed variance.')
print('If ranges overlap substantially, the improvement is real on average but any single')
print('seed/sample draw could look more ambiguous -- worth stating that nuance in the paper.')
